In [ ]:
# Cell 1 — Environment, paths, and configuration

import shutil
from pathlib import Path

# ------------------------------------------------------------
# Detect Google Colab
# ------------------------------------------------------------

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f"Environment: {'Google Colab' if IS_COLAB else 'Local Jupyter / Python'}")

# ------------------------------------------------------------
# Define the security_frameworks directory
# ------------------------------------------------------------

if IS_COLAB:
    SECURITY_FRAMEWORKS_DIR = Path("/content/security_frameworks")
    SECURITY_FRAMEWORKS_DIR.mkdir(parents=True, exist_ok=True)
else:
    # Local runs must be launched from inside security_frameworks/ so that
    # OSCAL_output/ lands next to this notebook rather than somewhere else.
    SECURITY_FRAMEWORKS_DIR = Path.cwd()
    if SECURITY_FRAMEWORKS_DIR.name != "security_frameworks":
        raise RuntimeError(
            "This notebook must be run with its working directory set to "
            "security_frameworks/. Current working directory: "
            f"{SECURITY_FRAMEWORKS_DIR}"
        )

# ------------------------------------------------------------
# Output location
# ------------------------------------------------------------
# NOTE: this notebook no longer clones usnistgov/OSCAL. An OSCAL/ directory
# left over from an earlier version of this notebook is not touched or read.

OUTPUT_DIR = SECURITY_FRAMEWORKS_DIR / "OSCAL_output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GITHUB_OWNER = "usnistgov"
GITHUB_REPO = "OSCAL"

# Pin a specific release tag (e.g. "v1.1.3") to reproduce an older schema set.
# Leave as None to resolve the latest published release.
PIN = None

print(f"Security frameworks directory: {SECURITY_FRAMEWORKS_DIR}")
print(f"Output directory:              {OUTPUT_DIR}")
print(f"PIN:                           {PIN!r}")


In [ ]:
# Cell 2 — Resolve the OSCAL release

import requests

API_TIMEOUT = 30  # seconds

if PIN is None:
    release_url = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/releases/latest"
else:
    release_url = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/releases/tags/{PIN}"

response = requests.get(
    release_url,
    headers={"Accept": "application/vnd.github+json"},
    timeout=API_TIMEOUT,
)
if response.status_code != 200:
    raise RuntimeError(
        f"Failed to resolve OSCAL release from {release_url}: "
        f"HTTP {response.status_code} {response.text[:500]}"
    )

release = response.json()

RELEASE_TAG = release["tag_name"]
RELEASE_PUBLISHED_AT = release["published_at"]
RELEASE_ASSETS = release["assets"]  # list of {name, browser_download_url, ...}

print("=" * 70)
print("OSCAL release resolved")
print("=" * 70)
print(f"Tag           : {RELEASE_TAG}")
print(f"Published at  : {RELEASE_PUBLISHED_AT}")
print(f"PIN requested : {PIN!r}")
print(f"Assets found  : {len(RELEASE_ASSETS)}")


In [ ]:
# Cell 3 — Download prebuilt JSON schemas from the release and write the manifest

import hashlib
import json
from datetime import datetime, timezone

DOWNLOAD_TIMEOUT = 60  # seconds

# Same set of models the old notebook produced via `make artifacts`.
MODELS = [
    "assessment-plan",
    "assessment-results",
    "catalog",
    "complete",
    "component",
    "mapping",
    "poam",
    "profile",
    "ssp",
]

ASSETS_BY_NAME = {asset["name"]: asset for asset in RELEASE_ASSETS}

TAG_DIR = OUTPUT_DIR / RELEASE_TAG
TAG_DIR.mkdir(parents=True, exist_ok=True)

JSON_FILES = {}
manifest_files = {}
missing_models = []

for model in MODELS:
    asset_name = f"oscal_{model}_schema.json"
    asset = ASSETS_BY_NAME.get(asset_name)

    if asset is None:
        print(f"✗ Missing release asset for model {model!r}: {asset_name}")
        missing_models.append({"model": model, "expected_asset": asset_name})
        continue

    download_url = asset["browser_download_url"]
    resp = requests.get(download_url, timeout=DOWNLOAD_TIMEOUT)
    if resp.status_code != 200:
        raise RuntimeError(
            f"Failed to download {asset_name} from {download_url}: "
            f"HTTP {resp.status_code}"
        )
    content = resp.content
    sha256 = hashlib.sha256(content).hexdigest()

    tag_dest = TAG_DIR / asset_name
    tag_dest.write_bytes(content)

    flat_dest = OUTPUT_DIR / asset_name
    flat_dest.write_bytes(content)

    stem = f"oscal_{model}_schema"
    JSON_FILES[stem] = flat_dest
    manifest_files[asset_name] = {
        "model": model,
        "source_url": download_url,
        "sha256": sha256,
        "tag_path": str(tag_dest),
        "flat_path": str(flat_dest),
    }
    print(f"✓ Downloaded {asset_name} (sha256={sha256[:12]}...)")

if not JSON_FILES:
    raise RuntimeError(
        f"No OSCAL JSON schema assets were found on release {RELEASE_TAG}. "
        f"Expected asset names like 'oscal_<model>_schema.json'."
    )

if missing_models:
    print(f"\n{len(missing_models)} model(s) missing from release {RELEASE_TAG}:")
    for m in missing_models:
        print(f"  - {m['model']} (expected {m['expected_asset']})")

# ------------------------------------------------------------
# Manifest
# ------------------------------------------------------------

manifest = {
    "repo": f"{GITHUB_OWNER}/{GITHUB_REPO}",
    "release_tag": RELEASE_TAG,
    "published_at": RELEASE_PUBLISHED_AT,
    "retrieved_at": datetime.now(timezone.utc).isoformat(),
    "pin": PIN,
    "files": manifest_files,
    "missing_models": missing_models,
}

manifest_path = OUTPUT_DIR / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 70)
print("OSCAL SCHEMAS DOWNLOADED FROM RELEASE")
print("=" * 70)
print(f"Release tag : {RELEASE_TAG}")
print(f"Schemas     : {len(JSON_FILES)} / {len(MODELS)} models")
print(f"Tag dir     : {TAG_DIR}")
print(f"Manifest    : {manifest_path}")


In [ ]:
# Cell 4 — Extract and print the inferred JSON structure

import json
from collections import OrderedDict

# ------------------------------------------------------------
# Schema inference utilities
# ------------------------------------------------------------

def json_type(value):
    if value is None: return "null"
    if isinstance(value, bool): return "boolean"
    if isinstance(value, int) and not isinstance(value, bool): return "integer"
    if isinstance(value, float): return "number"
    if isinstance(value, str): return "string"
    if isinstance(value, list): return "array"
    if isinstance(value, dict): return "object"
    return "unknown"

def merge_types(existing, new_type):
    if existing is None: return new_type
    existing_types = existing if isinstance(existing, list) else [existing]
    new_types = new_type if isinstance(new_type, list) else [new_type]
    merged = []
    for t in existing_types + new_types:
        if t not in merged: merged.append(t)
    return merged[0] if len(merged) == 1 else sorted(merged)

def schema_from_value(value):
    value_type = json_type(value)
    if value_type == "object":
        fields = OrderedDict()
        for key in sorted(value.keys()):
            fields[key] = schema_from_value(value[key])
        return {"type": "object", "fields": fields}
    if value_type == "array":
        if not value: return {"type": "array", "items": {}}
        item_schema = None
        for item in value:
            current = schema_from_value(item)
            if item_schema is None:
                item_schema = current
            elif item_schema.get("type") == "object" and current.get("type") == "object":
                item_schema = merge_object_schemas(item_schema, current)
            else:
                item_schema = merge_schema_nodes(item_schema, current)
        return {"type": "array", "items": item_schema}
    return {"type": value_type}

def merge_schema_nodes(a, b):
    a_type = a.get("type")
    b_type = b.get("type")
    if a_type == "object" and b_type == "object":
        return merge_object_schemas(a, b)
    if a_type == "array" and b_type == "array":
        if not a.get("items"): return b
        if not b.get("items"): return a
        return {"type": "array", "items": merge_schema_nodes(a["items"], b["items"])}
    return {"type": merge_types(a_type, b_type)}

def merge_object_schemas(a, b):
    merged_fields = OrderedDict()
    for key in sorted(set(a.get("fields", {})) | set(b.get("fields", {}))):
        if key in a.get("fields", {}) and key in b.get("fields", {}):
            merged_fields[key] = merge_schema_nodes(a["fields"][key], b["fields"][key])
        elif key in a.get("fields", {}):
            merged_fields[key] = a["fields"][key]
        else:
            merged_fields[key] = b["fields"][key]
    return {"type": "object", "fields": merged_fields}

def extract_schema(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return schema_from_value(data)

# ------------------------------------------------------------
# Analyze OSCAL Schema files
# ------------------------------------------------------------

schemas = OrderedDict()

for name, json_file in JSON_FILES.items():
    print(f"Analyzing: {json_file}")
    schemas[name] = extract_schema(json_file)

# ------------------------------------------------------------
# Print inferred structures
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INFERRED JSON STRUCTURE")
print("=" * 70)

print(
    json.dumps(
        schemas,
        indent=2,
        ensure_ascii=False,
    )
)